# 01 — Prompting: zero-shot and few-shot

This notebook builds prompts for token-level linguistic annotation. It can either call an API through a user-provided adapter or use a deterministic fallback that mimics realistic successes and failures.

Default mode is `USE_API = False`, so the notebook runs in Colab without secrets.

In [ ]:
!pip -q install jsonschema

In [ ]:
from pathlib import Path
import os, json, re, random, hashlib, datetime
import pandas as pd
import numpy as np

PROJECT_DIR = Path('/content/lrec2026_llm_annotator')
DATA_DIR = PROJECT_DIR / 'data' / 'sample'
SCHEMA_DIR = PROJECT_DIR / 'schemas'
OUTPUT_DIR = PROJECT_DIR / 'outputs'
for d in [DATA_DIR, SCHEMA_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

USE_API = False   # Change to True only after implementing/providing a safe provider adapter.
MODEL_NAME = 'fallback-mock-annotator-v0'
PROMPT_VERSION = 'v0.1'

print('USE_API =', USE_API)

## Load data

Run `00_setup_and_data.ipynb` first. If the data are missing, the next cell raises a clear error.

In [ ]:
csv_path = DATA_DIR / 'toy_sentences.csv'
if not csv_path.exists():
    raise FileNotFoundError('Run 00_setup_and_data.ipynb first to create toy_sentences.csv')

df = pd.read_csv(csv_path)
for col in ['tokens', 'gold_pos', 'gold_lemma', 'gold_features']:
    df[col] = df[col].apply(json.loads)

df[['id','language','script','domain','text','split']]

In [ ]:
UPOS = ['ADJ','ADP','ADV','AUX','CCONJ','DET','INTJ','NOUN','NUM','PART','PRON','PROPN','PUNCT','SCONJ','SYM','VERB','X']
FEATURES = ['Case', 'Number', 'Gender', 'Person', 'Tense', 'Mood', 'Voice']
CONFIDENCE = ['low', 'medium', 'high']

schema = json.loads((SCHEMA_DIR / 'pos_lemma_morph_schema.json').read_text(encoding='utf-8'))

## Prompt builder

The prompt fixes the task, the tokenisation policy, the allowed labels, and the expected JSON shape. Few-shot examples are selected from rows marked `split == "fewshot"`.

In [ ]:
def format_example(row):
    token_objs = []
    for surface, lemma, upos, feats in zip(row['tokens'], row['gold_lemma'], row['gold_pos'], row['gold_features']):
        token_objs.append({
            'surface': surface,
            'lemma': lemma,
            'upos': upos,
            'features': feats,
            'confidence': 'high',
            'comment': None
        })
    return json.dumps({
        'input': {'language': row['language'], 'sentence_id': row['id'], 'tokens': row['tokens']},
        'output': {'sentence_id': row['id'], 'language': row['language'], 'tokens': token_objs}
    }, ensure_ascii=False, indent=2)

fewshot_rows = df[df['split'] == 'fewshot'].copy()

def build_prompt(row, mode='zero_shot', max_examples=4):
    inventory = ', '.join(UPOS)
    prompt = f"""
You are assisting with linguistic annotation of historical and under-resourced languages.
Your task is to annotate the provided sentence token by token.

Return only valid JSON.
Do not translate the sentence.
Do not add, remove, split, merge, reorder, transliterate, or normalise tokens.
Use the provided UPOS inventory only: {inventory}.
Use feature names only from: {', '.join(FEATURES)}.
If unsure, keep the original token, use the best label you can, set confidence to "low", and add a short local comment.

Language: {row['language']}
Sentence ID: {row['id']}
Tokens: {json.dumps(row['tokens'], ensure_ascii=False)}
""".strip()
    if mode == 'few_shot':
        examples = []
        # Prefer same language example; then add others if needed.
        same_lang = fewshot_rows[fewshot_rows['language'] == row['language']]
        chosen = pd.concat([same_lang, fewshot_rows]).drop_duplicates('id').head(max_examples)
        for _, ex in chosen.iterrows():
            examples.append(format_example(ex))
        prompt += "\n\nFollow these validated examples:\n" + "\n\n".join(examples)
    prompt += "\n\nExpected JSON schema summary: sentence_id, language, tokens[{surface, lemma, upos, features, confidence, comment}]."
    return prompt

sample_row = df[df['split'] == 'eval'].iloc[0]
print(build_prompt(sample_row, mode='few_shot')[:2200])

## Provider adapter

The default tutorial path uses fallback predictions. If you want to call a model, implement `call_llm_api(prompt, schema)` for your chosen provider.

Keep the API key in an environment variable or Colab secret; never hard-code it in the notebook.

In [ ]:
def call_llm_api(prompt, schema=None):
    """Provider adapter placeholder.

    Implement this function for your chosen provider. It should return a raw string.
    Do not place API keys directly in the notebook.
    """
    raise NotImplementedError(
        'API mode is intentionally left as an adapter. Set USE_API=False or implement call_llm_api().'
    )

## Fallback annotator

This deterministic mock annotator is not a model. It creates plausible predictions with a few controlled failures so the validation and evaluation notebooks have something to analyse.

In [ ]:
def mock_prediction(row, mode='zero_shot'):
    tokens = list(row['tokens'])
    gold_pos = list(row['gold_pos'])
    gold_lemma = list(row['gold_lemma'])
    gold_features = list(row['gold_features'])

    out_tokens = []
    for i, surface in enumerate(tokens):
        upos = gold_pos[i]
        lemma = gold_lemma[i]
        feats = dict(gold_features[i])
        confidence = 'high'
        comment = None

        # Deterministic but interpretable perturbations.
        key = f"{row['id']}::{i}::{mode}"
        h = int(hashlib.md5(key.encode('utf-8')).hexdigest(), 16) % 100
        if mode == 'zero_shot':
            if h < 12 and upos == 'NOUN':
                upos = 'PROPN'
                confidence = 'medium'
                comment = 'Possible proper/common noun confusion.'
            elif h < 18 and upos == 'VERB':
                feats.pop('Tense', None)
                confidence = 'low'
                comment = 'Uncertain verbal morphology.'
            elif h < 22 and upos == 'DET':
                upos = 'ARTICLE'  # invalid on purpose
                confidence = 'medium'
                comment = 'Invalid label deliberately introduced.'
        else:  # few-shot: fewer errors
            if h < 6 and upos == 'NOUN':
                feats.pop('Case', None)
                confidence = 'medium'
                comment = 'Case uncertain.'

        out_tokens.append({
            'surface': surface,
            'lemma': lemma,
            'upos': upos,
            'features': feats,
            'confidence': confidence,
            'comment': comment
        })

    # One alignment failure in zero-shot mode.
    if mode == 'zero_shot' and row['id'] == 'xcl_002':
        out_tokens = out_tokens[:-1]

    pred = {'sentence_id': row['id'], 'language': row['language'], 'tokens': out_tokens}
    raw = json.dumps(pred, ensure_ascii=False)

    # One fenced response to test parser robustness.
    if mode == 'zero_shot' and row['id'] == 'oge_002':
        raw = '```json\n' + raw + '\n```'
    return raw

In [ ]:
def annotate_batch(data, mode='zero_shot'):
    records = []
    for _, row in data.iterrows():
        prompt = build_prompt(row, mode='few_shot' if mode == 'few_shot' else 'zero_shot')
        if USE_API:
            raw = call_llm_api(prompt, schema=schema)
            model_name = MODEL_NAME
        else:
            raw = mock_prediction(row, mode=mode)
            model_name = MODEL_NAME
        records.append({
            'sentence_id': row['id'],
            'language': row['language'],
            'split': row['split'],
            'mode': mode,
            'model_name': model_name,
            'prompt_version': PROMPT_VERSION,
            'prompt': prompt,
            'raw_response': raw,
            'timestamp': datetime.datetime.utcnow().isoformat() + 'Z'
        })
    return pd.DataFrame(records)

eval_df = df[df['split'].isin(['eval','unlabeled'])].copy()
zero_pred = annotate_batch(eval_df, mode='zero_shot')
few_pred = annotate_batch(eval_df, mode='few_shot')

for name, pred in [('zero_shot_raw.jsonl', zero_pred), ('few_shot_raw.jsonl', few_pred)]:
    path = OUTPUT_DIR / name
    pred.to_json(path, orient='records', lines=True, force_ascii=False)
    print('Wrote', path, len(pred), 'records')

zero_pred[['sentence_id','language','mode','raw_response']].head(3)

## Participant TODO

Change one part of the prompt and rerun only a small batch. Examples:

- add a stricter instruction about token alignment;
- add a few-shot example from the same language;
- require `confidence = low` when morphology is uncertain;
- remove the comments and see whether validation becomes easier.

Continue with `02_structured_outputs_and_validation.ipynb`.